# Chapter 6 — Pure Component Parameters: Scientific Figures

**Figures:**
1. Parameter sensitivity: vapor pressure vs $a_0$, $c_1$, $\varepsilon$, $\beta$
2. CPA parameter space landscape (2D contour)
3. Fitted vs experimental vapor pressure and liquid density
4. Comparison of parameter sets across different associating compounds

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim


NeqSim mode: pip


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
R = 8.314

## Figure 6.1 — CPA Parameter Comparison Across Compounds

In [3]:
# CPA parameters from literature (Kontogeorgis & Folas, 2010)
compounds = ["Water", "Methanol", "Ethanol", "MEG", "DEG", "TEG", "Acetic acid"]
schemes =   ["4C",    "2B",       "2B",      "4C",  "4C",  "4C",  "1A"]
eps_over_R = [1017.3,  2899.5,     2589.8,    2375.0, 2660.0, 2848.0, 3044.4]  # K
beta_vals =  [0.0692,  0.0162,     0.0080,    0.0141, 0.0092, 0.0065, 0.0750]  # -
b_vals =     [1.45,    3.10,       4.91,      5.14,   7.90,   10.63,  4.69]    # x10^-5 m3/mol

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.0, 3.0))

# Left: eps/R vs beta
colors_pt = [BLUE, ORANGE, GREEN, PURPLE, PINK, LIME, "#636363"]
for i, (comp, eps_r, beta_v, scheme) in enumerate(zip(compounds, eps_over_R, beta_vals, schemes)):
    ax1.scatter(beta_v, eps_r, color=colors_pt[i], s=60, zorder=5, edgecolors="black", linewidths=0.5)
    ax1.annotate(f"{comp}\n({scheme})", xy=(beta_v, eps_r),
                 xytext=(beta_v + 0.003, eps_r + 50), fontsize=6)

ax1.set_xlabel(r"$\beta$ (bonding volume)")
ax1.set_ylabel(r"$\varepsilon/R$ (K)")
ax1.set_title("(a) Association parameters")
ax1.grid(True, alpha=0.3)

# Right: b vs number of sites
n_sites = [4, 2, 2, 4, 4, 4, 1]
for i, (comp, b_v, ns_v) in enumerate(zip(compounds, b_vals, n_sites)):
    ax2.scatter(b_v, ns_v, color=colors_pt[i], s=60, zorder=5, edgecolors="black", linewidths=0.5)
    ax2.annotate(comp, xy=(b_v, ns_v), xytext=(b_v + 0.2, ns_v + 0.1), fontsize=6)

ax2.set_xlabel(r"$b \times 10^{5}$ (m$^3$/mol)")
ax2.set_ylabel("Number of association sites")
ax2.set_title("(b) Co-volume vs sites")
ax2.set_yticks([1, 2, 3, 4])
ax2.grid(True, alpha=0.3)

fig.tight_layout()
save(fig, "fig_ch06_01_parameter_comparison.png")

Saved: fig_ch06_01_parameter_comparison.png


## Figure 6.2 — Parameter Sensitivity: Vapor Pressure

In [4]:
# Show how vapor pressure changes with eps/R and beta for a model 2B fluid
# Simplified: association shifts effective attraction
# Higher eps => lower vapor pressure (stronger H-bonds in liquid)
# Higher beta => lower vapor pressure

eps_range = np.linspace(500, 3500, 50)
beta_range = np.linspace(0.005, 0.10, 50)
Eps_grid, Beta_grid = np.meshgrid(eps_range, beta_range)

# Simplified model: ln(Pvap) ~ -eps*beta*rho (proportional to association strength)
T_ref = 373.15  # K (100 C)
rho_ref = 50000.0  # mol/m3
b_ref = 3.0e-5
eta_ref = b_ref * rho_ref / 4.0
g_ref = 1.0 / (1.0 - 1.9 * eta_ref)

# Association contribution to reduced pressure
Delta_grid = g_ref * (np.exp(Eps_grid * R / (R * T_ref)) - 1.0) * b_ref * Beta_grid
# Degree of bonding
rd_grid = rho_ref * Delta_grid
XA_grid = (-1.0 + np.sqrt(1.0 + 4.0 * rd_grid)) / (2.0 * rd_grid + 1e-30)
bonding = 1.0 - XA_grid

fig, ax = plt.subplots(figsize=(3.5, 3.0))
cs = ax.contourf(Eps_grid, Beta_grid * 100, bonding * 100, levels=15,
                  cmap="YlOrRd")
cbar = fig.colorbar(cs, ax=ax, label="Degree of bonding (%)")

# Mark known compounds
ax.plot(1017.3, 6.92, 'ws', markersize=7, markeredgecolor="black", zorder=5)
ax.annotate("Water", xy=(1017.3, 6.92), xytext=(1200, 7.5), fontsize=7, color="white",
            fontweight="bold")
ax.plot(2899.5, 1.62, 'w^', markersize=7, markeredgecolor="black", zorder=5)
ax.annotate("MeOH", xy=(2899.5, 1.62), xytext=(2600, 2.5), fontsize=7, color="white",
            fontweight="bold")

ax.set_xlabel(r"$\varepsilon / R$ (K)")
ax.set_ylabel(r"$\beta \times 100$")
ax.set_title("Association parameter landscape")
fig.tight_layout()
save(fig, "fig_ch06_02_parameter_landscape.png")

Saved: fig_ch06_02_parameter_landscape.png


## Figure 6.3 — NeqSim: CPA Water Vapor Pressure Accuracy

In [5]:
if NEQSIM_MODE == "pip":
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
else:
    from neqsim import jneqsim
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# NIST reference data
nist_T = [25, 50, 75, 100, 125, 150, 175, 200, 250, 300, 350]
nist_Pvap = [0.03169, 0.1235, 0.3858, 1.014, 2.321, 4.760, 8.926, 15.55, 39.76, 85.88, 165.3]

cpa_Pvap = []
for T_C in nist_T:
    try:
        f = SystemSrkCPAstatoil(273.15 + float(T_C), 1.0)
        f.addComponent("water", 1.0)
        f.setMixingRule(10)
        ops = ThermodynamicOperations(f)
        ops.bubblePointPressureFlash(False)
        cpa_Pvap.append(float(f.getPressure("bara")))
    except Exception:
        cpa_Pvap.append(np.nan)

# Relative deviation
dev_pct = [(c - n) / n * 100 for c, n in zip(cpa_Pvap, nist_Pvap)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.0, 3.0))
ax1.semilogy(nist_T, nist_Pvap, 'ko', markersize=5, label="NIST")
ax1.semilogy(nist_T, cpa_Pvap, color=BLUE, linewidth=1.5, label="CPA (NeqSim)")
ax1.set_xlabel("Temperature (\u00b0C)"); ax1.set_ylabel("Vapor pressure (bar)")
ax1.set_title("(a) Vapor pressure"); ax1.legend(frameon=True); ax1.grid(True, alpha=0.3, which="both")

ax2.bar(nist_T, dev_pct, width=15, color=BLUE, alpha=0.7)
ax2.axhline(y=0, color="black", linewidth=0.5)
ax2.set_xlabel("Temperature (\u00b0C)"); ax2.set_ylabel("Deviation (%)")
ax2.set_title("(b) CPA vs NIST deviation"); ax2.grid(True, alpha=0.3)

fig.tight_layout()
save(fig, "fig_ch06_03_cpa_water_pvap_accuracy.png")

Saved: fig_ch06_03_cpa_water_pvap_accuracy.png
